In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "PAQ_F.xpt"
csv_file = DATA_DIR / "PAQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 PAQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PAQ706",
    "PAD590",
    "PAD600",
    "PAQ605",
    "PAQ610",
    "PAD615",
    "PAQ620",
    "PAQ625",
    "PAD630",
    "PAQ635",
    "PAQ640",
    "PAD645",
    "PAQ650",
    "PAQ655",
    "PAD660",
    "PAQ665",
    "PAQ670",
    "PAD675",
    "PAD680",
    "PAAQUEX"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "PAQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 9771:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 21:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,771 rows x 21 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Never globally replace this value before validating:
#
#     variable
#     count
#     CDC valid range / meaning
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached PAQ_F contains exactly:
#
# PAQ706 = 86
# PAD590 = 285
# PAD600 = 764
# PAD680 = 2
#
#
# CDC:
#
# PAQ706:
#     0-7 days is a valid range.
#
# PAD590:
#     0 = Less than 1 hour
#     CDC count = 285
#
# PAD600:
#     0 = Less than 1 hour
#     CDC count = 764
#
# PAD680:
#     Valid range = 0-1200 minutes.
#
# Therefore these tiny values are validated zeros.
#

EXPECTED_TINY_COUNTS = {
    "PAQ706": 86,
    "PAD590": 285,
    "PAD600": 764,
    "PAD680": 2
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated PAQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 1137:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Exact 1,137-value tiny-zero pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

#
# No global replacement.
#
# Only the exact validated variables/positions
# are restored to numeric zero.
#

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 1137:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# XPT SHOULD contain.
#
# Only:
#
#     validated TINY_VALUE -> 0
#
# positions differ from the raw pandas XPT.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CHECK GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# This check MUST occur before Int64 conversion.
#
# After validated zero restoration, the attached
# PAQ_F contains NO genuine fractional values.
#
# If a different file/copy contains fractions,
# STOP instead of rounding them.
#
# This same safeguard protects genuine:
#
#     Dietary decimals
#     Lab measurements
#     Examination measurements
#     Sample weights
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in PAQ_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT PROVEN WHOLE-NUMBER FIELDS WITHOUT .0
# ============================================================

#
# After:
#
#   validated zero restoration
#   genuine decimal validation
#
# all 21 PAQ_F fields are proven whole-number values.
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All PAQ_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CHECK - PAQ706
# ============================================================

#
# Days physically active at least 60 minutes.
#
# CDC:
#
# 0-7 = 2,224
# 77  = Refused = 1
# 99  = Don't know = 3
# Missing = 7,543
#
# Attached XPT contains 86 encoded-zero artifacts.
#

s = df["PAQ706"]


print("\n--- PAQ706 ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-7 count:",
    int(
        s.between(
            0,
            7
        ).sum()
    )
)


if not (
    int(
        s.between(
            0,
            7
        ).sum()
    )
    == 2224
    and
    int(
        s.eq(0)
        .sum()
    )
    == 86
    and
    int(
        s.eq(77)
        .sum()
    )
    == 1
    and
    int(
        s.eq(99)
        .sum()
    )
    == 3
    and
    int(
        s.isna()
        .sum()
    )
    == 7543
):

    raise ValueError(
        "PAQ706 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: PAQ706 matches CDC."
)


# ============================================================
# 12. CDC CHECK - PAD590
# ============================================================

#
# Hours watch TV/videos.
#
# CDC:
#
# 0 = Less than 1 hour = 285
# 1 = 1 hour           = 539
# 2 = 2 hours          = 734
# 3 = 3 hours          = 320
# 4 = 4 hours          = 159
# 5 = 5+ hours         = 158
# 6 = None             = 31
# 77 = Refused         = 1
# 99 = Don't know      = 1
# Missing              = 7,543
#

EXPECTED_PAD590 = {
    0: 285,
    1: 539,
    2: 734,
    3: 320,
    4: 159,
    5: 158,
    6: 31,
    77: 1,
    99: 1
}


actual = (
    df["PAD590"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_PAD590
    or
    int(
        df["PAD590"]
        .isna()
        .sum()
    )
    != 7543
):

    raise ValueError(
        "PAD590 differs from CDC."
    )


print(
    "PASS: PAD590 matches CDC."
)


# ============================================================
# 13. CDC CHECK - PAD600
# ============================================================

#
# Hours use computer / computer games.
#
# CDC:
#
# 0 = Less than 1 hour = 764
# 1 = 1 hour           = 407
# 2 = 2 hours          = 164
# 3 = 3 hours          = 64
# 4 = 4 hours          = 20
# 5 = 5+ hours         = 35
# 6 = None             = 773
# 77 = Refused         = 1
# 99 = Don't know      = 0
# Missing              = 7,543
#

EXPECTED_PAD600 = {
    0: 764,
    1: 407,
    2: 164,
    3: 64,
    4: 20,
    5: 35,
    6: 773,
    77: 1
}


actual = (
    df["PAD600"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_PAD600
    or
    int(
        df["PAD600"]
        .isna()
        .sum()
    )
    != 7543
):

    raise ValueError(
        "PAD600 differs from CDC."
    )


print(
    "PASS: PAD600 matches CDC."
)


# ============================================================
# 14. CDC YES/NO ACTIVITY CHECKS
# ============================================================

CDC_ACTIVITY_FLAGS = {

    "PAQ605": {
        "values": {
            1: 1236,
            2: 6263,
            9: 1
        },
        "missing": 2271
    },

    "PAQ620": {
        "values": {
            1: 2537,
            2: 4962,
            9: 1
        },
        "missing": 2271
    },

    "PAQ635": {
        "values": {
            1: 2247,
            2: 5252,
            9: 1
        },
        "missing": 2271
    },

    "PAQ650": {
        "values": {
            1: 1892,
            2: 5607,
            9: 1
        },
        "missing": 2271
    },

    "PAQ665": {
        "values": {
            1: 2977,
            2: 4521,
            9: 2
        },
        "missing": 2271
    }
}


print("\n--- CDC ACTIVITY FLAG CHECKS ---")


for col, expected in CDC_ACTIVITY_FLAGS.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected['missing']}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 15. CDC CHECK - DAYS PER WEEK VARIABLES
# ============================================================

#
# All are valid from 1-7 days.
#

DAY_CHECKS = {

    "PAQ610": {
        "range_count": 1235,
        "dontknow": 1,
        "missing": 8535
    },

    "PAQ625": {
        "range_count": 2534,
        "dontknow": 3,
        "missing": 7234
    },

    "PAQ640": {
        "range_count": 2247,
        "dontknow": 0,
        "missing": 7524
    },

    "PAQ655": {
        "range_count": 1892,
        "dontknow": 0,
        "missing": 7879
    },

    "PAQ670": {
        "range_count": 2975,
        "dontknow": 2,
        "missing": 6794
    }
}


print("\n--- CDC DAY COUNT CHECKS ---")


for col, expected in DAY_CHECKS.items():

    s = df[col]


    range_count = int(
        s.between(
            1,
            7
        ).sum()
    )


    refused = int(
        s.eq(77)
        .sum()
    )


    dontknow = int(
        s.eq(99)
        .sum()
    )


    missing = int(
        s.isna()
        .sum()
    )


    if range_count != expected["range_count"]:

        raise ValueError(
            f"{col}: valid-range count differs from CDC."
        )


    if refused != 0:

        raise ValueError(
            f"{col}: unexpected Refused value."
        )


    if dontknow != expected["dontknow"]:

        raise ValueError(
            f"{col}: Don't know count differs from CDC."
        )


    if missing != expected["missing"]:

        raise ValueError(
            f"{col}: missing count differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 16. CDC CHECK - ACTIVITY MINUTE VARIABLES
# ============================================================

#
# These fields were collected as time but are
# released as whole-number MINUTES.
#
# They are not decimal variables in PAQ_F.
#

MINUTE_CHECKS = {

    "PAD615": {
        "min": 10,
        "max": 960,
        "range_count": 1229,
        "dontknow": 3,
        "missing": 8539
    },

    "PAD630": {
        "min": 10,
        "max": 1080,
        "range_count": 2527,
        "dontknow": 3,
        "missing": 7241
    },

    "PAD645": {
        "min": 10,
        "max": 960,
        "range_count": 2239,
        "dontknow": 5,
        "missing": 7527
    },

    "PAD660": {
        "min": 10,
        "max": 480,
        "range_count": 1889,
        "dontknow": 1,
        "missing": 7881
    },

    "PAD675": {
        "min": 10,
        "max": 600,
        "range_count": 2969,
        "dontknow": 2,
        "missing": 6800
    }
}


print("\n--- CDC ACTIVITY MINUTE CHECKS ---")


for col, expected in MINUTE_CHECKS.items():

    s = df[col]


    range_count = int(
        s.between(
            expected["min"],
            expected["max"]
        ).sum()
    )


    refused = int(
        s.eq(7777)
        .sum()
    )


    dontknow = int(
        s.eq(9999)
        .sum()
    )


    missing = int(
        s.isna()
        .sum()
    )


    if range_count != expected["range_count"]:

        raise ValueError(
            f"{col}: range count differs from CDC."
        )


    if refused != 0:

        raise ValueError(
            f"{col}: unexpected Refused value."
        )


    if dontknow != expected["dontknow"]:

        raise ValueError(
            f"{col}: Don't know count differs from CDC."
        )


    if missing != expected["missing"]:

        raise ValueError(
            f"{col}: missing count differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 17. CDC CHECK - PAD680
# ============================================================

#
# Minutes sedentary activity.
#
# CDC:
#
# 0-1200 = 7,476
# 7777   = 0
# 9999   = 18
# Missing = 2,277
#
# Attached raw XPT contains 2 tiny-value zeros.
#

s = df["PAD680"]


print("\n--- PAD680 ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-1200 count:",
    int(
        s.between(
            0,
            1200
        ).sum()
    )
)


if not (
    int(
        s.between(
            0,
            1200
        ).sum()
    )
    == 7476
    and
    int(
        s.eq(0)
        .sum()
    )
    == 2
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 18
    and
    int(
        s.isna()
        .sum()
    )
    == 2277
):

    raise ValueError(
        "PAD680 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: PAD680 matches CDC."
)


# ============================================================
# 18. CDC CHECK - PAAQUEX
# ============================================================

#
# Questionnaire source flag for weighting.
#
# CDC:
#
# 1 = Sample Person Interview = 9,119
# 2 = MEC CAPI               =   652
# Missing                    =     0
#

EXPECTED_PAAQUEX = {
    1: 9119,
    2: 652
}


actual = (
    df["PAAQUEX"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_PAAQUEX
    or
    int(
        df["PAAQUEX"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "PAAQUEX differs from CDC."
    )


print(
    "PASS: PAAQUEX matches CDC."
)


# ============================================================
# 19. VALIDATED ZERO COUNTS
# ============================================================

ZERO_COUNTS = {
    "PAQ706": 86,
    "PAD590": 285,
    "PAD600": 764,
    "PAD680": 2
}


print("\n--- VALIDATED ZERO CHECK ---")


for col, expected_zero_count in ZERO_COUNTS.items():

    actual_zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col}: {actual_zero_count:,} zeros"
    )


    if actual_zero_count != expected_zero_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_zero_count:,} zeros, "
            f"found {actual_zero_count:,}."
        )


print(
    "PASS: All corrected zero counts are correct."
)


# ============================================================
# 20. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected differences from raw pandas XPT:
#
# PAQ706:
#     86 tiny values -> 0
#
# PAD590:
#     285 tiny values -> 0
#
# PAD600:
#     764 tiny values -> 0
#
# PAD680:
#     2 tiny values -> 0
#
# TOTAL = 1,137
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 21 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 21. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# All PAQ_F variables have been proven to be
# whole-number fields in this release.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 22. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 23. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 21 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 24. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 25. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 26. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# PAQ_F contains NO genuine decimal variables.
#
# Therefore ".0" should not appear anywhere in
# this particular final CSV.
#
# IMPORTANT:
#
# DO NOT use this same global check on Dietary,
# Lab, Examination, or another file containing
# genuine decimal variables.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "PAQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "PAQ706 zero restorations:",
    86
)


print(
    "PAD590 zero restorations:",
    285
)


print(
    "PAD600 zero restorations:",
    764
)


print(
    "PAD680 zero restorations:",
    2
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 PAQ_F XPT -> CSV
Rows: 9,771
Columns: 21
PASS: Structure = 9,771 rows x 21 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 1,137

Tiny values by variable:
PAD600    764
PAD590    285
PAQ706     86
PAD680      2

PASS: Exact 1,137-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
PAQ706: 86 restored to 0
PAD590: 285 restored to 0
PAD600: 764 restored to 0
PAD680: 2 restored to 0

PASS: 1,137 validated zero values restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in PAQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All PAQ_F fields formatted without unnecessary .0.

--- PAQ706 ---
Zero: 86
0-7 count: 2224
PASS: PAQ706 matches CDC.
PASS: PAD590 matches CDC.
PASS: PAD600 matches CDC.

--- CDC ACTIVITY FLAG CHECKS ---
PASS: PAQ605 matches CDC.
PASS: PAQ620 matches CDC.
PASS: PAQ635 matches CDC.
PASS: PAQ650 matches CDC.
PASS: PAQ665 m